In [1]:
import json
from os import getenv

from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import (
    ChatCompletionMessage,
    ChatCompletionMessageParam,
    ChatCompletionToolParam,
)

load_dotenv()

True

In [2]:
# 1.定义工具
def get_weather(location:str) -> str:
    return f"{location} 当前天气为【晴天】"

def square_root(x: float) -> float:
    return x ** 0.5

# 2.组织工具字典
tools_by_name = {
    "get_weather": get_weather,
    "square_root": square_root,
}

# 3.定义工具的描述
tools:list[ChatCompletionToolParam] = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather of a location, the user should supply a location first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"]
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "square_root",
            "description": "Calculate the square root of a given number.",
            "parameters": {
                "type": "object",
                "properties": {
                    "x": {
                        "type": "number",
                        "description": "The number to calculate square root for",
                    }
                },
                "required": ["x"]
            },
        }
    },
]

# 4.创建一个 LLM 客户端
client = OpenAI(
    api_key=getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

In [3]:
# 5.定义发消息的方法
def send_message(messages: list[ChatCompletionMessageParam]) -> ChatCompletionMessage:
    """
    发送消息
    """
    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=messages,
        tools=tools,
        extra_body={
            "thinking":{
                "type" :"disabled"
            }
        }

    )
    return response.choices[0].message

# 6.执行 tool 调用的方法
def execute_tool(message, messages):
    """解析 Tool 信息，执行对应的函数，并将结果追加到消息列表"""
    tool_call = message.tool_calls[0]
    tool_id = tool_call.id
    tool_name = tool_call.function.name
    tool_args = json.loads(tool_call.function.arguments)

    print(f"调用工具: {tool_name}, 参数: {tool_args}")

    # 根据函数名找到对应函数并执行
    result = tools_by_name[tool_name](**tool_args)
    print(f"工具结果: {result}")

    # 将 AI 消息和 Tool 结果追加到消息列表
    messages.append(message)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_id,
        "content": result
    })
    return messages

In [ ]:
# 7.主要的流程
# 准备用户信息
messages: list[ChatCompletionMessageParam] = [{"role": "user", "content": "苏州今天天气怎么样？"}]
print(f"【User】: {messages[0]['content']}") # type: ignore

# 第一次调用 llm，获取 tool call
message = send_message(messages)
print(f"【Tool】: {message.tool_calls[0]}") # type: ignore

# 执行Tool并将结果传回LLM
messages = execute_tool(message, messages)

# 再次调用LLM，生成最终答案
message = send_message(messages)
print(f"【AI】: {message.content}")

【User】: 苏州今天天气怎么样？
【Tool】: ChatCompletionMessageFunctionToolCall(id='call_00_NUgslo29Gowx76fQ2vE00922', function=Function(arguments='{"location": "苏州, 江苏"}', name='get_weather'), type='function', index=0)
调用工具: get_weather, 参数: {'location': '苏州, 江苏'}
工具结果: 苏州, 江苏 当前天气为【晴天】
【AI】: 苏州今天天气是**晴天** ☀️，适合出门活动。如果需要了解具体气温或未来几天的预报，可以再告诉我。
